# 16 — FPR-Regularized Fine-tuning

**Research question:** Can we reduce identity-probe false-positive rate (FPR)
without destroying accuracy, using an explicit FPR penalty in the loss?

The masking experiment (§3.4) showed that identity bias is a single-token
lexical shortcut, and naive training-time masking hurts recall. A smarter
approach: add a differentiable **FPR penalty** during second-stage fine-tuning:

$$\mathcal{L} = \text{BCE}(\text{task batch}) + \gamma \cdot \frac{1}{|P|}\sum_{x \in P} \sigma(f(x)_{\text{toxic}})$$

where $P$ is the 230-sentence identity probe set and $\sigma$ is the sigmoid.
The BCE term preserves accuracy on the main task; the FPR penalty pushes
the model to predict probe sentences as non-toxic.

**Protocol:**
1. Start from the trained `distilbert-student` (FPR = 0.062, F1 = 0.772)
2. Sweep gamma ∈ {0.1, 0.5, 1.0, 2.0}
3. For each, train 2 epochs on 20k Jigsaw examples + FPR penalty
4. Measure F1, FPR, and identity-hate recall

## Setup
1. Runtime → **GPU** (Colab T4, ~5 min per gamma)
2. Ensure `src/`, `data/`, and `models/distilbert-student/` are available
3. Run all cells

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT / "DeepProjectHate" / "src").exists():
    ROOT = ROOT / "DeepProjectHate"
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())

In [ ]:
from device import get_device
print("device:", get_device())

from paths import SPLIT_INDICES_PATH, LOCAL_DISTILBERT_STUDENT_DIR
assert SPLIT_INDICES_PATH.exists(), "Run scripts/make_splits.py first"
assert LOCAL_DISTILBERT_STUDENT_DIR.exists(), "Need trained distilbert-student weights"
print("Student weights:", LOCAL_DISTILBERT_STUDENT_DIR)

## 1. Gamma sweep training

In [ ]:
GAMMAS = [0.1, 0.5, 1.0, 2.0]

for g in GAMMAS:
    print(f"\n{'='*60}")
    print(f"Training gamma={g}")
    print(f"{'='*60}")
    !python scripts/train_fpr_finetune.py --gamma {g} --epochs 2 --max-train-samples 20000

## 2. Evaluate all variants

In [ ]:
import numpy as np
import torch
from pathlib import Path

from distill import DistilBertStudentScorer
from fairness import build_identity_probe_df, per_group_fpr, control_fpr
from splits import get_split_frame
from paths import LOCAL_DISTILBERT_STUDENT_DIR, MODELS_DIR, TOXIC_LABELS
from sklearn.metrics import f1_score, recall_score

probe_df = build_identity_probe_df()
test_df = get_split_frame("test").sample(5000, random_state=211)
test_texts = test_df["comment_text"].astype(str).tolist()
test_toxic = test_df["toxic"].values
ih_mask = test_df["identity_hate"] == 1
ih_texts = test_df.loc[ih_mask, "comment_text"].astype(str).tolist()
ih_labels = np.ones(len(ih_texts), dtype=int)

results = []

# Baseline student (gamma=0)
variants = [("baseline (gamma=0)", str(LOCAL_DISTILBERT_STUDENT_DIR))]
for g in GAMMAS:
    d = MODELS_DIR / f"distilbert-student-fpr-g{g}"
    if d.exists():
        variants.append((f"gamma={g}", str(d)))
    else:
        print(f"[SKIP] {d} not found")

for name, path in variants:
    print(f"\nEvaluating {name} ...")
    scorer = DistilBertStudentScorer(model_path=path)
    
    # Test F1
    probs = scorer.predict_toxic_proba(test_texts)
    preds = (probs >= 0.5).astype(int)
    f1 = f1_score(test_toxic, preds)
    
    # Identity FPR
    probe_probs = scorer.predict_toxic_proba(probe_df["text"].tolist())
    identity_fpr = float((probe_probs >= 0.5).mean())
    
    # Identity-hate recall
    ih_probs = scorer.predict_toxic_proba(ih_texts)
    ih_recall = float((ih_probs >= 0.5).mean())
    
    row = {"variant": name, "path": path,
           "f1": round(f1, 4), "identity_fpr": round(identity_fpr, 4),
           "ih_recall": round(ih_recall, 4)}
    results.append(row)
    print(f"  F1={f1:.4f}  FPR={identity_fpr:.4f}  IH_recall={ih_recall:.4f}")

print("\nDone.")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
display(df)

## 3. Tradeoff plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

gammas_plot = [0] + GAMMAS

# Left: F1 vs gamma
ax = axes[0]
ax.plot(gammas_plot, df["f1"].values, "o-", color="steelblue", linewidth=2)
ax.set_xlabel(r"$\gamma$ (FPR penalty weight)")
ax.set_ylabel("Test F1 (toxic)")
ax.set_title("Accuracy vs. FPR penalty")
ax.axhline(df["f1"].iloc[0], ls="--", color="gray", alpha=0.5, label="baseline")
ax.legend()
ax.grid(True, alpha=0.3)

# Center: FPR vs gamma
ax = axes[1]
ax.plot(gammas_plot, df["identity_fpr"].values, "s-", color="crimson", linewidth=2)
ax.set_xlabel(r"$\gamma$ (FPR penalty weight)")
ax.set_ylabel("Identity probe FPR")
ax.set_title("Fairness vs. FPR penalty")
ax.axhline(0, ls="--", color="gray", alpha=0.5)
ax.grid(True, alpha=0.3)

# Right: FPR vs F1 (Pareto front)
ax = axes[2]
ax.scatter(df["f1"].values, df["identity_fpr"].values, s=80, c="darkorange", zorder=5)
for _, row in df.iterrows():
    ax.annotate(row["variant"], (row["f1"], row["identity_fpr"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("Test F1 (toxic)")
ax.set_ylabel("Identity probe FPR")
ax.set_title("F1 vs. FPR tradeoff")
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig("report/figures/fpr_finetune_tradeoff.png", dpi=150, bbox_inches="tight")
print("Saved report/figures/fpr_finetune_tradeoff.png")
plt.show()

## 4. Save results

In [ ]:
out = {"variants": results, "gammas": [0] + GAMMAS}
out_path = Path("data/processed/fpr_finetune_results.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved {out_path}")

## 5. Download results

Run this cell in Colab to download the results locally.

In [ ]:
try:
    from google.colab import files
    files.download("data/processed/fpr_finetune_results.json")
    files.download("report/figures/fpr_finetune_tradeoff.png")
except ImportError:
    print("Not on Colab; files are saved locally.")